# Konsolidierung: Arbeitsmappen und Adjudikation nach CoNLL

**Ordnerkontext:** liegt in `00_werkzeuge/`, liest aus
`02_quelldaten/`, `03_annotation_laufend/` und `05_ergebnisse/iaa/`,
schreibt nach `04_konsolidiert/`.

## Konfiguration


In [ ]:
from pathlib import Path

BASIS_QUELL  = Path("..") / "02_quelldaten"
BASIS_ARBEIT = Path("..") / "03_annotation_laufend"
BASIS_IAA    = Path("..") / "05_ergebnisse" / "iaa"
ZIEL         = Path("..") / "04_konsolidiert"

CONLL = BASIS_QUELL / "goldstandard_v1.conll"
LAUF  = None        
ZIEL.mkdir(parents=True, exist_ok=True)
assert CONLL.exists(), f"{CONLL} nicht gefunden."

ERWARTET = {"ner": 4, "pos": 3}

AUSGESCHLOSSEN = {"_ausgegeben"}


def mappen_finden(ebene, stamm):
    """Sucht Arbeitsmappen beider Tabellenformate.

    Die Bearbeitenden geben je nach Programm .xlsx oder .ods zurueck;
    pandas liest beide, sofern openpyxl beziehungsweise odfpy vorliegt.
    Temporaerdateien von Excel beginnen mit ~$ und werden uebergangen.
    """
    treffer = []
    for endung in ("xlsx", "ods"):
        treffer += BASIS_ARBEIT.glob(f"{ebene}/**/{stamm}.{endung}")
    return sorted(p for p in treffer
                  if not p.name.startswith("~$")
                  and not AUSGESCHLOSSEN & set(p.parts))


pos_mappen = mappen_finden("pos", "*_pos_tranche*")
ner_mappen = mappen_finden("ner", "*_ner_tranche*")

def divergenzen_finden(praefix):
    dateien = sorted(BASIS_IAA.glob(f"divergenzen_{praefix}_t*_lauf*.csv"))
    if LAUF is not None:
        dateien = [p for p in dateien if f"lauf{LAUF:02d}" in p.name]
    else:
        #je Tranche den niedrigsten Lauf waehlen
        beste = {}
        for p in dateien:
            tranche = p.name.split("_t")[1].split("_")[0]
            if tranche not in beste or p.name < beste[tranche].name:
                beste[tranche] = p
        dateien = sorted(beste.values())
    return dateien

pos_div = divergenzen_finden("pos")
ner_div = divergenzen_finden("ner")

print(f"Wortartenebene: {len(pos_mappen)} Mappen, "
      f"{len(pos_div)} Divergenzliste(n)")
for p in pos_mappen: print("   ", p.relative_to(BASIS_ARBEIT))
for p in pos_div:    print("    ->", p.name)
print(f"\nFigurenebene:   {len(ner_mappen)} Mappen, "
      f"{len(ner_div)} Divergenzliste(n)")
for p in ner_mappen: print("   ", p.relative_to(BASIS_ARBEIT))
for p in ner_div:    print("    ->", p.name)

Wortartenebene: 3 Mappen, 1 Divergenzliste(n)
    pos/tranche1/A4_pos_tranche1_final.xlsx
    pos/tranche1/A5_pos_tranche1_final.xlsx
    pos/tranche1/A6_pos_tranche1_final.ods
    -> divergenzen_pos_t1_lauf01.csv

Figurenebene:   8 Mappen, 2 Divergenzliste(n)
    ner/tranche1/A1_ner_tranche1_final.xlsx
    ner/tranche1/A3_ner_tranche1_final.ods
    ner/tranche1/A7_ner_tranche1_final.xlsx
    ner/tranche1/A8_ner_tranche1_final.xlsx
    ner/tranche2/A1_ner_tranche2_final.xlsx
    ner/tranche2/A3_ner_tranche2_final.ods
    ner/tranche2/A7_ner_tranche2_final.xlsx
    ner/tranche2/A8_ner_tranche2_final.xlsx
    -> divergenzen_ner_t1_lauf01.csv
    -> divergenzen_ner_t2_lauf01.csv


## Lesepakete und Vollständigkeit prüfen


In [ ]:
from collections import defaultdict
import importlib

#1) Lesepakete: pandas liest .xlsx ueber openpyxl und .ods ueber odfpy
benoetigt = {"xlsx": ("openpyxl", "openpyxl"), "ods": ("odf", "odfpy")}
gebraucht = {p.suffix.lstrip(".") for p in pos_mappen + ner_mappen}
for endung in sorted(gebraucht):
    modul, paket = benoetigt[endung]
    if importlib.util.find_spec(modul) is None:
        raise ImportError(
            f"Fuer .{endung}-Mappen wird '{paket}' benoetigt: "
            f"pip install {paket}")
    print(f"  .{endung}: {paket} vorhanden")

#2) Vollständigkeit je Ebene und Tranche
def tranche_von(pfad):
    """Liest die Tranchennummer aus dem Dateinamen."""
    teil = pfad.stem.split("tranche")
    return teil[1] if len(teil) > 1 else "?"

print()
unvollstaendig = []
for ebene, mappen in (("ner", ner_mappen), ("pos", pos_mappen)):
    if not mappen:
        print(f"{ebene}: keine Mappen gefunden — Ebene wird uebersprungen.")
        continue
    je_tranche = defaultdict(set)
    for p in mappen:
        je_tranche[tranche_von(p)].add(p.name.split("_")[0])
    for t in sorted(je_tranche):
        personen = sorted(je_tranche[t])
        soll = ERWARTET[ebene]
        marke = "ok" if len(personen) == soll else "UNVOLLSTAENDIG"
        print(f"{ebene} Tranche {t}: {len(personen)} von {soll} "
              f"Bearbeitungen ({', '.join(personen)}) — {marke}")
        if len(personen) != soll:
            unvollstaendig.append((ebene, t, personen, soll))

if unvollstaendig:
    print()
    for ebene, t, personen, soll in unvollstaendig:
        print(f"  {ebene} Tranche {t}: erwartet {soll}, "
              f"gefunden {len(personen)}: {', '.join(personen)}")
    raise RuntimeError(
        "Unvollstaendige Bearbeitungen. Entweder fehlen Ruecklaeufe, oder "
        "ERWARTET in Schritt 0 ist an die tatsaechliche Besetzung "
        "anzupassen. Ohne vollstaendige Mappen wuerden strittige Token "
        "faelschlich als einstimmig ins Gold uebernommen.")

print()
print("Vollstaendig — die Konsolidierung kann beginnen.")

  .ods: odfpy vorhanden
  .xlsx: openpyxl vorhanden

ner Tranche 1_final: 4 von 4 Bearbeitungen (A1, A3, A7, A8) — ok
ner Tranche 2_final: 4 von 4 Bearbeitungen (A1, A3, A7, A8) — ok
pos Tranche 1_final: 3 von 3 Bearbeitungen (A4, A5, A6) — ok

Vollstaendig — die Konsolidierung kann beginnen.


## Schritt 1 — Struktur der Vorlage einlesen


In [3]:
zeilen_struktur = []      # ("kommentar"|"leer"|"token", Inhalt)
token_ids, token_texte = [], []
lauf_id = 0

for z in CONLL.read_text(encoding="utf-8").splitlines():
    if z.startswith("#"):
        zeilen_struktur.append(("kommentar", z))
    elif not z.strip():
        zeilen_struktur.append(("leer", ""))
    else:
        token = z.split("\t")[0]
        zeilen_struktur.append(("token", token))
        token_ids.append(lauf_id)
        token_texte.append(token)
        lauf_id += 1

print(f"Vorlage: {len(zeilen_struktur)} Zeilen, davon {len(token_ids)} Token")
print(f"Kennungen laufen von 0 bis {token_ids[-1]}")

Vorlage: 4490 Zeilen, davon 4137 Token
Kennungen laufen von 0 bis 4136


## Schritt 2 — Hilfsfunktionen


In [ ]:
import pandas as pd

def mappen_lesen(pfade, blatt, wertspalte):
    """Liest alle Mappen und gibt {id: {person: wert}} zurueck."""
    werte = {}
    personen = set()
    for pfad in pfade:
        person = pfad.name.split("_")[0]
        personen.add(person)
        df = pd.read_excel(pfad, sheet_name=blatt, dtype=str).fillna("")
        for _, r in df.iterrows():
            i = int(r["id"])
            werte.setdefault(i, {})[person] = str(r[wertspalte]).strip()
    return werte, sorted(personen)


def entscheidungen_lesen(pfade, schluesselspalten):
    """Liest die adjudizierten Divergenzlisten.
    Rueckgabe: (entschieden, offen) als Dictionaries ueber den Schluessel."""
    entschieden, offen = {}, []
    for pfad in pfade:
        df = pd.read_csv(pfad, dtype=str).fillna("")
        for _, r in df.iterrows():
            schluessel = tuple(int(r[s]) for s in schluesselspalten)
            wert = str(r.get("entscheidung", "")).strip()
            if wert:
                entschieden[schluessel] = wert
            else:
                offen.append((pfad.name, schluessel,
                              str(r.get("token", r.get("text", "")))))
    return entschieden, offen


def conll_schreiben(ziel, spalte2, spalte3):
    """Schreibt strukturerhaltend; spalte2/spalte3 sind Listen ueber Token."""
    aus, k = [], 0
    for typ, inhalt in zeilen_struktur:
        if typ == "token":
            aus.append(f"{inhalt}\t{spalte2[k]}\t{spalte3[k]}")
            k += 1
        elif typ == "kommentar":
            aus.append(inhalt)
        else:
            aus.append("")
    ziel.write_text("\n".join(aus) + "\n", encoding="utf-8")
    #Gegenprobe: Tokenfolge unverändert?
    kontrolle = [z.split("\t")[0] for z in ziel.read_text(encoding="utf-8")
                 .splitlines() if z.strip() and not z.startswith("#")]
    assert kontrolle == token_texte, "Tokenfolge veraendert!"
    return len(aus)


print("Hilfsfunktionen definiert.")

Hilfsfunktionen definiert.


## Schritt 3 — Wortartenebene konsolidieren



In [5]:
pos_gold = None
if not pos_mappen:
    print("Keine Wortarten-Mappen gefunden — Schritt uebersprungen.")
else:
    werte, personen = mappen_lesen(pos_mappen, "POS", "tag")
    print(f"Bearbeitende: {personen} | Token mit Eintraegen: {len(werte)}")

    entschieden, offen = entscheidungen_lesen(pos_div, ["id"])
    print(f"Adjudikationsentscheidungen: {len(entschieden)} "
          f"| ohne Entscheidung: {len(offen)}")

    fehlend, einstimmig, adjudiziert, konflikt = [], 0, 0, []
    tags = []
    for i in token_ids:
        eintraege = werte.get(i, {})
        vorhanden = [v for v in eintraege.values() if v]
        if len(vorhanden) < len(personen) or not vorhanden:
            fehlend.append(i)
            tags.append("_")
            continue
        if len(set(vorhanden)) == 1:
            tags.append(vorhanden[0]); einstimmig += 1
        elif (i,) in entschieden:
            tags.append(entschieden[(i,)]); adjudiziert += 1
        else:
            konflikt.append((i, token_texte[i], eintraege))
            tags.append("_")

    print(f"\nEinstimmig:   {einstimmig}")
    print(f"Adjudiziert:  {adjudiziert}")
    print(f"Nicht bearbeitet: {len(fehlend)}")
    print(f"Ungeloeste Konflikte: {len(konflikt)}")
    for i, tok, e in konflikt[:10]:
        print(f"    id {i} '{tok}': " + ", ".join(f"{p}={v}"
                                                 for p, v in e.items()))
    if konflikt:
        raise RuntimeError(
            f"{len(konflikt)} Divergenzen ohne Entscheidung. Bitte die "
            f"Spalte 'entscheidung' in den Divergenzlisten fuellen.")
    if fehlend:
        print("\nHINWEIS: Nicht bearbeitete Token erhalten '_'. Das ist "
              "nur zulaessig,")
        print("solange die betreffende Tranche noch laeuft.")
    pos_gold = tags

Bearbeitende: ['A4', 'A5', 'A6'] | Token mit Eintraegen: 1794
Adjudikationsentscheidungen: 245 | ohne Entscheidung: 0

Einstimmig:   1549
Adjudiziert:  245
Nicht bearbeitet: 2343
Ungeloeste Konflikte: 0

HINWEIS: Nicht bearbeitete Token erhalten '_'. Das ist nur zulaessig,
solange die betreffende Tranche noch laeuft.


## Schritt 4 — Figurenebene konsolidieren


In [ ]:
ner_gold = None
if not ner_mappen:
    print("Keine Figuren-Mappen gefunden — Schritt uebersprungen.")
else:
    werte, personen = mappen_lesen(ner_mappen, "NER", "markierung")
    print(f"Bearbeitende: {personen} | Token mit Eintraegen: {len(werte)}")

    entschieden, offen = entscheidungen_lesen(ner_div, ["id_start", "id_ende"])
    print(f"Adjudikationsentscheidungen: {len(entschieden)} "
          f"| ohne Entscheidung: {len(offen)}")

    marken = []
    einstimmig, strittig_token, unbearbeitet = 0, 0, 0
    for i in token_ids:
        eintraege = werte.get(i, {})
        vorhanden = [v for v in eintraege.values() if v]
        if len(vorhanden) < len(personen):
            #Token einer noch nicht bearbeiteten Tranche
            marken.append("O"); unbearbeitet += 1
        elif len(set(vorhanden)) == 1:
            marken.append(vorhanden[0]); einstimmig += 1
        else:
            marken.append("O"); strittig_token += 1

    gesetzt, verworfen, unbekannt = 0, 0, []
    sortiert = sorted(entschieden.items(),
                      key=lambda p: str(p[1]).strip().lower()
                      in ("ja", "j", "figur", "b-figur"))
    for (start, ende), wert in sortiert:
        w = wert.strip().lower()
        if w in ("ja", "j", "figur", "b-figur"):
            marken[start] = "B-FIGUR"
            for k in range(start + 1, ende + 1):
                marken[k] = "I-FIGUR"
            gesetzt += 1
        elif w in ("nein", "n", "o", "keine figur"):
            for k in range(start, ende + 1):
                marken[k] = "O"
            verworfen += 1
        else:
            unbekannt.append(((start, ende), wert))

    print(f"\nEinstimmige Token: {einstimmig}")
    print(f"Strittige Token (Ausgang O, dann adjudiziert): {strittig_token}")
    print(f"Noch nicht bearbeitet (andere Tranche): {unbearbeitet}")
    print(f"Adjudiziert gesetzt: {gesetzt} | verworfen: {verworfen}")
    if unbekannt:
        print("Unverstaendliche Entscheidungen (erwartet 'ja' oder 'nein'):")
        for s, w in unbekannt[:10]:
            print(f"    Spanne {s}: {w!r}")
        raise RuntimeError("Unverstaendliche Eintraege in 'entscheidung'.")
    if offen:
        print(f"\n{len(offen)} strittige Spanne(n) ohne Entscheidung:")
        for name, s, text in offen[:10]:
            print(f"    {name} Spanne {s}: {text!r}")
        raise RuntimeError("Bitte die Spalte 'entscheidung' fuellen.")

    for (start, ende), wert in entschieden.items():
        if str(wert).strip().lower() in ("ja", "j", "figur", "b-figur"):
            assert marken[start] == "B-FIGUR", (
                f"Spanne {start}-{ende} wurde ueberschrieben; "
                f"vorgefunden: {marken[start]}")

    repariert = 0
    for k, m in enumerate(marken):
        if m == "I-FIGUR" and (k == 0 or marken[k - 1] == "O"):
            marken[k] = "B-FIGUR"; repariert += 1
    if repariert:
        print(f"\nBIO-Reparatur: {repariert} freistehendes I-FIGUR "
              f"zu B-FIGUR gemacht.")
    ner_gold = marken

Bearbeitende: ['A1', 'A3', 'A7', 'A8'] | Token mit Eintraegen: 4137
Adjudikationsentscheidungen: 77 | ohne Entscheidung: 0

Einstimmige Token: 4064
Strittige Token (Ausgang O, dann adjudiziert): 73
Noch nicht bearbeitet (andere Tranche): 0
Adjudiziert gesetzt: 28 | verworfen: 49


## Schritt 5 — Goldfassungen schreiben


In [7]:
GROB = {"ADJA": "ADJ", "ADJD": "ADJ"}
geschrieben = []

if pos_gold:
    ziel = ZIEL / "goldstandard_v1_pos_final.conll"
    conll_schreiben(ziel, pos_gold, ["O"] * len(token_ids))
    geschrieben.append(ziel)

    grob = [GROB.get(t, t) for t in pos_gold]
    ziel_grob = ZIEL / "goldstandard_v1_pos_final_upos.conll"
    conll_schreiben(ziel_grob, grob, ["O"] * len(token_ids))
    geschrieben.append(ziel_grob)

    from collections import Counter
    c = Counter(pos_gold)
    print(f"Wortarten fein: ADJA={c.get('ADJA', 0)}, ADJD={c.get('ADJD', 0)}, "
          f"uebrige {sum(v for k, v in c.items() if k not in ('ADJA','ADJD'))}")
    print(f"Wortarten grob: ADJ={c.get('ADJA', 0) + c.get('ADJD', 0)}")

if ner_gold:
    ziel = ZIEL / "goldstandard_v1_ner_final.conll"
    conll_schreiben(ziel, ["_"] * len(token_ids), ner_gold)
    geschrieben.append(ziel)
    n_fig = sum(1 for m in ner_gold if m == "B-FIGUR")
    print(f"Figuren: {n_fig} Spannen")

print()
for z in geschrieben:
    print(f"  geschrieben und geprueft: {z}")
if not geschrieben:
    print("  Nichts geschrieben — keine Mappen vorhanden.")

Wortarten fein: ADJA=65, ADJD=33, uebrige 4039
Wortarten grob: ADJ=98
Figuren: 202 Spannen

  geschrieben und geprueft: ../04_konsolidiert/goldstandard_v1_pos_final.conll
  geschrieben und geprueft: ../04_konsolidiert/goldstandard_v1_pos_final_upos.conll
  geschrieben und geprueft: ../04_konsolidiert/goldstandard_v1_ner_final.conll
